# Agent Executor + Parallel Search [MCP]

Example how to give an agent executor web access through the [Parallel Search MCP] server.

The server is reachable over Streamable HTTP at `https://search.parallel.ai/mcp` and publishes two tools:

* `web_search` - run one or more search queries and get back LLM friendly excerpts
* `web_fetch`  - read the content of specific URLs

It accepts **anonymous requests**, so this notebook needs no API key. Anonymous traffic is rate limited
and meant for evaluation; add an API key through the `x-api-key` header for anything heavier.
Using the server is subject to the [Parallel customer terms] and [privacy policy].

> **Disclosure**: this how-to was contributed by George Pickett, who works at Parallel.

[MCP]: https://github.com/modelcontextprotocol
[Parallel Search MCP]: https://docs.parallel.ai/integrations/mcp/search-mcp
[Parallel customer terms]: https://parallel.ai/customer-terms
[privacy policy]: https://parallel.ai/privacy-policy

In [ ]:
var userHomeDir = System.getProperty("user.home");
var localRespoUrl = "file://" + userHomeDir + "/.m2/repository/";
var langchain4jVersion = "1.22.0";
var langchain4jbeta = "1.22.0-beta32";
var langgraph4jVersion = "1.9.4";

Remove installed package from Jupiter cache

In [ ]:
%%bash
rm -rf \{userHomeDir}/Library/Jupyter/kernels/rapaio-jupyter-kernel/mima_cache/org/bsc/langgraph4j

add local maven repo and install dependencies

In [ ]:
%dependency /add-repo local \{localRespoUrl} release|never snapshot|always
// %dependency /list-repos
%dependency /add org.slf4j:slf4j-jdk14:2.0.9
%dependency /add org.bsc.langgraph4j:langgraph4j-core:\{langgraph4jVersion}
%dependency /add org.bsc.langgraph4j:langgraph4j-langchain4j:\{langgraph4jVersion}
%dependency /add org.bsc.langgraph4j:langgraph4j-agent-executor:\{langgraph4jVersion}
%dependency /add dev.langchain4j:langchain4j:\{langchain4jVersion}
%dependency /add dev.langchain4j:langchain4j-mcp:\{langchain4jbeta}
%dependency /add dev.langchain4j:langchain4j-ollama:\{langchain4jVersion}

%dependency /resolve

**Initialize Logger**

In [ ]:
try( var file = new java.io.FileInputStream("./logging.properties")) {
    java.util.logging.LogManager.getLogManager().readConfiguration( file );
}

var log = org.slf4j.LoggerFactory.getLogger("ParallelSearch");

## Setup MCP client

`StreamableHttpMcpTransport` talks to the remote server over plain HTTP, so unlike the
[mcp/postgres] example there is nothing to install and no Docker container to start.

No `Authorization` or `x-api-key` header is sent: the client connects anonymously.

[mcp/postgres]: https://github.com/modelcontextprotocol/servers/tree/main/src/postgres

In [ ]:
import dev.langchain4j.mcp.client.DefaultMcpClient;
import dev.langchain4j.mcp.client.transport.http.StreamableHttpMcpTransport;

import java.time.Duration;
import java.util.List;
import java.util.Map;

var transport = new StreamableHttpMcpTransport.Builder()
    .url( "https://search.parallel.ai/mcp" )
    .timeout( Duration.ofSeconds(60) )
    .logRequests(false) // set to true if you want to see the traffic in the log
    .logResponses(false)
    .build();

var mcpClient = new DefaultMcpClient.Builder()
    .transport( transport )
    .clientName( "langgraph4j-how-tos" )
    .clientVersion( "1.0" )
    .initializationTimeout( Duration.ofSeconds(30) )
    .toolExecutionTimeout( Duration.ofMinutes(2) )
    .build();

// the tools published by the server
mcpClient.listTools()
    .stream()
    .map( tool -> "%s - %s".formatted( tool.name(), tool.description().lines().findFirst().orElse("") ) )
    .toList();

## Setup Agent Executor

Only `web_search` and `web_fetch` are registered, so the agent keeps the same tool set even if
the server starts publishing more of them.

A failing tool call is reported back to the model as text instead of breaking the graph: the model
can then retry with a different query or answer without the tool.

In [ ]:
import org.bsc.langgraph4j.agentexecutor.AgentExecutor;

import dev.langchain4j.model.ollama.OllamaChatModel;

var webTools = List.of( "web_search", "web_fetch" );

final var model = OllamaChatModel.builder()
    .baseUrl( "http://localhost:11434" )
    .temperature(0.0)
    .logRequests(true)
    .logResponses(true)
    .timeout( Duration.ofMinutes(5) )
    .modelName("qwen2.5-coder:latest")
    .build();

var agentBuilder = AgentExecutor.builder()
                        .chatModel(model);

for( var toolSpecification : mcpClient.listTools() ) {
    if( !webTools.contains( toolSpecification.name() ) ) continue;

    agentBuilder.tool( toolSpecification, ( request, memoryId) -> {
        var result = mcpClient.executeTool( request );
        if( result.isError() ) {
            log.warn( "{} failed: {}", request.name(), result.resultText() );
        }
        return result.resultText();
    });
}

var agent = agentBuilder.build().compile();

## Ask a question that needs the web

The model decides when to search and when to fetch a page. The answer is built from the tool
results, so it can quote sources instead of relying on what the model memorized.

In [ ]:
import dev.langchain4j.data.message.ChatMessage;
import dev.langchain4j.data.message.SystemMessage;
import dev.langchain4j.data.message.UserMessage;

import org.bsc.langgraph4j.GraphInput;
import org.bsc.langgraph4j.RunnableConfig;

var messages = List.<ChatMessage>of(
    SystemMessage.from("""
        You are a web research assistant.
        Use web_search to look things up and web_fetch only when you need the full text of a page.
        Answer using the tool results and cite the source URLs.
        """),
    UserMessage.from( "What is langgraph4j and which checkpoint savers does it provide?" )
);

agent.invoke( GraphInput.args( Map.of( "messages", messages ) ), RunnableConfig.empty() )
    .flatMap(AgentExecutor.State::finalResponse)
    .orElse("no response");

Release the HTTP connection when the agent is no longer needed

In [ ]:
mcpClient.close();